# 01 - AIMU

This is the first notebook in the **Agent Frameworks** series. Every notebook in the series
implements the *same* scientific use case so you can compare frameworks side by side; only the
framework changes. See [README.md](../README.md) for the full series and the shared design.

## The use case: agentic literature research

Given a research question, the agent:

1. **Searches live sources** (medRxiv + bioRxiv preprints via the Europe PMC keyword API),
2. **Confirms relevance** of each hit and saves only the matches into a **local document store**
   (a curated, reusable corpus, gated so off-topic papers never land),
3. **Synthesizes** a cited answer from the curated corpus, and
4. **Evaluates and loops**: a critic reviews the synthesis and feeds gaps back for another round.

Everything runs on a **local model** through [Ollama](https://ollama.com).

## What AIMU brings

[`aimu`](https://github.com/) wraps a model client in an agentic loop (`Agent`), provides a
native `DocumentStore`, and composes workflows like the generate-evaluate loop with
`EvaluatorOptimizer`. Tools are plain Python functions decorated with `@aimu.tool`.

> **Script version:** [`scripts/01_aimu.py`](../scripts/01_aimu.py) runs this same workflow from the command line.

## 01 - Setup

Requires a running Ollama server with a tool-capable model and an embedding model:

```bash
ollama pull qwen3.6:27b
ollama pull nomic-embed-text
```

`qwen3.6:27b` is large but reliably handles the multi-step tool use this task needs. A smaller
model such as `qwen3.5:9b` runs faster but is less consistent at open-ended research.

In [ ]:
import os

import dotenv

import aimu
from aimu.agents import Agent, EvaluatorOptimizer
from aimu.memory import DocumentStore

from genscai import research, paths

dotenv.load_dotenv(paths.root / ".env")

MODEL = f"ollama:{os.environ.get('GENSCAI_AGENT_MODEL', 'qwen3.6:27b')}"

RESEARCH_QUESTION = (
    "What interventions and control strategies have recent preprints proposed or evaluated for "
    "dengue outbreaks, and what evidence do they report?"
)

## 02 - The shared research tools

`genscai.research` provides the framework-agnostic search/fetch functions used by *every* notebook
in this series. They query medRxiv and bioRxiv preprints through the
[Europe PMC](https://europepmc.org/) keyword API (which indexes both servers and returns abstracts
reliably). A quick look at what one search returns:

In [ ]:
hits = research.search_medrxiv("dengue vaccination", max_results=3)
for h in hits:
    print(h["date"], "-", h["title"][:80])
    print("   ", h["doi"])

## 03 - The document store and agent tools

AIMU's `DocumentStore` is the *native* store for this notebook (persistent, file-backed). We expose
three tools to the agent. The relevance decision is the agent's own reasoning: it reads each
abstract and calls `save_relevant_paper` **only** for matches, so the store stays curated.

`search_preprints` caches each hit by DOI so the agent can save a paper with just its DOI, without
echoing long abstracts back through the model.

In [ ]:
store = DocumentStore(persist_path=str(paths.output / "agent_frameworks" / "aimu_store"))

# Cache of search hits so a paper can be saved by DOI alone.
_seen: dict[str, dict] = {}


@aimu.tool
def search_preprints(query: str) -> str:
    """Search medRxiv and bioRxiv preprints. Returns each hit's DOI, title, date, and abstract."""
    results = research.search_medrxiv(query, max_results=5) + research.search_biorxiv(query, max_results=3)
    if not results:
        return "No results found."
    blocks = []
    for article in results:
        _seen[article["doi"]] = article
        blocks.append(
            f"DOI: {article['doi']}\nTitle: {article['title']}\nDate: {article['date']}\n"
            f"Abstract: {(article['abstract'] or '')[:600]}"
        )
    return "\n\n".join(blocks)


@aimu.tool
def save_relevant_paper(doi: str) -> str:
    """Save a paper you have confirmed is relevant to the research question, identified by its DOI."""
    article = _seen.get(doi)
    if not article:
        return f"Unknown DOI {doi}. Search for it first so its details are available."
    content = (
        f"# {article['title']}\n\nDOI: {doi}\nAuthors: {article['authors']}\n"
        f"Date: {article['date']}\nURL: {article['url']}\n\n{article['abstract']}"
    )
    store.write(f"/{doi.replace('/', '_')}.md", content)
    return f"Saved: {article['title']}"


@aimu.tool
def read_saved_papers() -> str:
    """Read every paper saved in the local document store, to ground your synthesis."""
    saved = store.list_paths()
    if not saved:
        return "No papers saved yet."
    return "\n\n---\n\n".join(store.read(p) for p in saved)

## 04 - The researcher agent

An `Agent` is a model client wrapped in a loop: it calls tools until it produces a turn with no
tool calls (or hits `max_iterations`). We stream the run to watch it search, gate relevance, store
matches, and synthesize. `final_answer_prompt` guarantees a written synthesis even if the loop hits
its iteration cap mid-tool-use.

In [ ]:
researcher = Agent(
    aimu.client(MODEL),
    system_message=(
        "You are a research assistant building a cited literature review grounded in a curated corpus.\n"
        "Workflow:\n"
        "1. Call read_saved_papers first to see what is already curated.\n"
        "2. Use search_preprints to find candidate papers. Always include the key topic term from the "
        "question (e.g. 'dengue') in every search query so results stay on-topic.\n"
        "3. For EACH candidate, judge whether its abstract is relevant to the question. If it is plausibly "
        "relevant, call save_relevant_paper(doi); err toward saving rather than discarding. Skip only "
        "clearly off-topic hits.\n"
        "4. Before writing any synthesis you MUST call read_saved_papers again, and base the synthesis "
        "ONLY on the papers it returns, citing each by title and DOI. The store already contains relevant "
        "papers, so never claim no papers were found. Be concise."
    ),
    tools=[search_preprints, save_relevant_paper, read_saved_papers],
    max_iterations=12,
    reset_messages_on_run=True,
    final_answer_prompt=(
        "Call read_saved_papers, then write the final cited synthesis based only on those saved papers. "
        "Do not claim there are no papers if the store is non-empty."
    ),
    name="researcher",
)


def stream_run(runner, task):
    """Print an AIMU agent/workflow run, marking tool calls and streaming text."""
    for chunk in runner.run(task, stream=True):
        if chunk.is_tool_call():
            print(f"\n  [tool] {chunk.content}")
        elif chunk.is_text():
            print(chunk.content, end="")
    print()


stream_run(researcher, RESEARCH_QUESTION)

## 05 - Add an evaluator: the self-correcting loop

`EvaluatorOptimizer` runs a generator, has a critic review the output, and loops with the critic's
feedback until the critic replies `PASS` or `max_rounds` is reached. The document store persists
across rounds, so each revision builds on the corpus curated so far.

In [ ]:
critic = Agent(
    aimu.client(MODEL),
    system_message=(
        "You review a literature synthesis for an infectious-disease researcher. The synthesis is built "
        "from a curated corpus of saved preprints. Reply with exactly PASS if it answers the question, "
        "cites specific papers (title + DOI), and reports evidence from them. Only reply REVISE: <specific "
        "gap> when something concrete is missing. Do NOT ask for topics outside the saved corpus, and do "
        "not penalize a focused answer. Prefer PASS when the synthesis is reasonable."
    ),
    max_iterations=1,
    reset_messages_on_run=True,
    name="critic",
)

review = EvaluatorOptimizer(generator=researcher, evaluator=critic, max_rounds=2, pass_keyword="PASS")

synthesis = review.run(RESEARCH_QUESTION)
print(synthesis)

## 06 - Inspect the curated corpus

The agent built a local, reusable corpus of vetted papers. Because the store is gated by the
relevance check, every document here was judged relevant to the question.

In [ ]:
print("Saved papers:")
for path in store.list_paths():
    print(" ", path)

print("\nFirst saved paper:\n")
print(store.read(store.list_paths()[0]))

## 07 - AIMU notes

- **Tools**: plain functions with `@aimu.tool`; the docstring becomes the tool description and type
  hints become the schema. Lowest-ceremony tool definition of the frameworks in this series.
- **Document store**: `aimu.memory.DocumentStore` is built in and persistent. AIMU also ships
  `make_memory_tools`/`make_retrieval_tool` (in `aimu.tools.builtin`) and a vector-backed
  `SemanticMemoryStore` if you want semantic retrieval instead of a curated file store.
- **The feedback loop**: `EvaluatorOptimizer` expresses generate→evaluate→revise as a first-class
  workflow object; you supply two agents and a pass keyword.
- **Local models**: `aimu.client("ollama:...")` is all that's needed; the agentic loop, streaming,
  and tool-calling work the same across providers.